# 第 6 课：记忆的应用 — Prompt 注入与 RAG

> 学完这节课，你会：理解 RAG 的基本原理、知道 Prompt 注入的两种主要方式、掌握分层注入策略、能写出 `build_prompt()` 函数。

---

In [ ]:
!pip install openai python-dotenv -q

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()

api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print(f"API Key 已加载（前8位: {api_key[:8]}...）")
else:
    print("⚠️ 未找到 OPENAI_API_KEY，请在课程目录创建 .env 文件")

## 6.1 RAG 是什么

**RAG = Retrieval Augmented Generation = 检索增强生成**

听起来很高大上，其实你每天都在做类似的事情：

| 你写代码 | AI 用 RAG |
|---------|----------|
| 遇到 bug → Google 搜索 | 收到用户消息 → 从记忆库检索 |
| 看到 StackOverflow 答案 | 拿到相关的记忆片段 |
| 根据答案修改代码 | 把记忆塞进 prompt，生成回复 |

**核心思路：先检索（R），再生成（G），检索到的内容增强（A）了生成质量。**

### RAG 流程图

```
用户发消息: "周末去爬山吗？"
         │
         ▼
┌─────────────────────┐
│  ① 检索 Retrieval   │
│  从记忆库搜索相关信息  │
└──────────┬──────────┘
           │  搜到："小明喜欢爬山，上次一起爬了香山"
           ▼
┌─────────────────────┐
│  ② 增强 Augment     │
│  把检索结果塞进 prompt │
└──────────┬──────────┘
           │  system: "联系人小明，喜欢爬山，上次去了香山..."
           │  user: "帮我回复：周末去爬山吗？"
           ▼
┌─────────────────────┐
│  ③ 生成 Generate    │
│  LLM 生成回复        │
└──────────┬──────────┘
           │
           ▼
AI 回复: "好啊！上次香山爬得不错，这次去哪？"
```

### 没有 RAG vs 有 RAG

我们用代码体验一下差别：

In [ ]:
# 没有 RAG：AI 什么都不知道
response_no_rag = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "帮我回复这条消息：'周末去爬山吗？'"}
    ]
)
print("【没有 RAG】")
print(response_no_rag.choices[0].message.content)
print()

In [ ]:
# 有 RAG：先"检索"到相关记忆，再注入 prompt
retrieved_memory = """联系人：小明
关系：大学同学，关系很好
爱好：喜欢户外运动，尤其是爬山和跑步
历史：你们经常一起运动，上次一起爬了香山"""

response_with_rag = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": f"""你是一个帮用户生成聊天回复的助手。语气自然随意，像朋友聊天。

以下是关于这个联系人的记忆信息：
{retrieved_memory}

请生成 3 条回复建议。"""
        },
        {"role": "user", "content": "帮我回复：'周末去爬山吗？'"}
    ]
)
print("【有 RAG】")
print(response_with_rag.choices[0].message.content)

看到区别了吗？有记忆信息时，AI 会提到"香山"、"跑步"这些具体细节，回复更像是你自己说的话。

---

## 6.2 Prompt 注入的基本方式

记忆信息怎么"塞"进 prompt？主要有两种方式：

### 方式一：全量注入 System Prompt

把所有相关信息**一股脑塞进** system prompt。ChatGPT 的 Memory 功能就是这么干的。

```
优点：简单粗暴，不需要检索逻辑
缺点：信息量大时浪费 token，无关信息多
适用：记忆量小（比如只有几十条 fact）
```

### 方式二：检索后注入（经典 RAG）

先根据当前消息**检索**最相关的记忆片段，只把相关的部分注入。

```
优点：token 用量可控，信息高度相关
缺点：需要额外的检索系统（向量数据库等）
适用：记忆量大（成百上千条记忆）
```

In [ ]:
# ===== 模拟记忆库 =====
all_memories = {
    "小明": [
        "大学同学，计算机系",
        "喜欢爬山和跑步",
        "上次一起爬了香山",
        "在字节跳动工作",
        "养了一只叫花花的猫",
        "最近在学吉他",
        "不吃香菜",
        "喜欢看科幻电影",
        "生日是 3 月 15 号",
        "老家在成都",
    ]
}


# ===== 方式一：全量注入 =====
def inject_all(contact, user_message):
    """把这个联系人的所有记忆全部塞进 system prompt"""
    memories = all_memories.get(contact, [])
    memory_text = "\n".join(f"- {m}" for m in memories)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"""你是帮用户回复消息的助手。语气自然随意。

关于联系人 {contact} 的所有记忆：
{memory_text}

请生成 3 条回复建议。"""
            },
            {"role": "user", "content": f"帮我回复：'{user_message}'"}
        ]
    )
    return response.choices[0].message.content


# ===== 方式二：检索后注入（简单关键词匹配模拟） =====
def inject_retrieved(contact, user_message):
    """先检索相关记忆，只注入相关的部分"""
    memories = all_memories.get(contact, [])

    # 简单的关键词匹配模拟检索（实际系统会用向量相似度）
    keywords = user_message.lower().split()
    relevant = [
        m for m in memories
        if any(kw in m for kw in keywords + ["关系", "同学"])
    ]
    # 如果没检索到，至少带上基本信息
    if not relevant:
        relevant = memories[:2]

    memory_text = "\n".join(f"- {m}" for m in relevant)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"""你是帮用户回复消息的助手。语气自然随意。

关于联系人 {contact} 的相关记忆：
{memory_text}

请生成 3 条回复建议。"""
            },
            {"role": "user", "content": f"帮我回复：'{user_message}'"}
        ]
    )
    return response.choices[0].message.content


print("===== 方式一：全量注入 =====")
print(inject_all("小明", "周末去爬山吗"))
print()
print("===== 方式二：检索后注入 =====")
print(inject_retrieved("小明", "周末去爬山吗"))

两种方式对比：
- **全量注入**把"不吃香菜"、"生日 3 月 15 号"这些跟爬山无关的信息也塞进去了
- **检索后注入**只保留了"爬山"、"香山"相关的记忆

当记忆只有 10 条时差别不大，但如果有 500 条记忆呢？全量注入就扛不住了。

---

## 6.3 注入多少信息才合适？

这是一个非常实际的问题。信息量有个"甜蜜点"：

```
太少 ←————————— 甜蜜点 ———————————→ 太多
AI 回复没针对性    回复精准且省钱     token 浪费、性能下降
```

### "Lost in the Middle" 效应

研究发现：当注入大量信息时，AI 对**中间部分**的信息关注度最低。头尾的信息利用率最高，中间的容易被"忽略"。

这就像你看一篇超长的文档——开头和结尾记得最清楚，中间的内容经常划过去了。

In [ ]:
# 实验：不同信息量对回复质量的影响

# 100 字左右的记忆
memory_short = "联系人小明，大学同学，喜欢爬山。"

# 500 字左右的记忆
memory_medium = """联系人：小明
关系：大学同学，计算机系，关系很好，经常一起吃饭运动。
性格：开朗外向，喜欢组织活动。
爱好：喜欢户外运动，尤其是爬山和跑步。每周至少跑步三次。
运动经历：上次一起爬了香山，走的是野路线，用了 4 个小时。之前还一起爬过泰山。
工作：在字节跳动做后端开发，最近项目很忙。
饮食：不吃香菜，喜欢吃川菜，经常推荐成都小吃。
近况：最近在学吉他，想组个乐队。养了一只猫叫花花。"""

# 2000 字左右的记忆（大量细节）
memory_long = """联系人：小明
关系：大学同学，2018年入学，计算机科学与技术专业，同一个实验室。关系很好，是最亲密的朋友之一。
大学经历：一起做过课程项目，做了一个校园二手交易平台。一起参加过 ACM 竞赛，拿了省赛铜奖。经常一起在图书馆自习。毕业旅行一起去了云南，去了大理和丽江。
性格：开朗外向，喜欢组织活动，经常是聚会的发起人。做事比较靠谱。有时候会迟到。
爱好：喜欢户外运动，尤其是爬山和跑步。每周至少跑步三次，目标是跑完全马。去年跑了北京半马，成绩 1 小时 52 分。
爬山经历：上次一起爬了香山，走的是野路线，用了 4 个小时。之前还一起爬过泰山（看了日出）和华山（走的长空栈道）。计划今年一起去爬四姑娘山。
工作：在字节跳动做后端开发，负责推荐系统。最近项目很忙，经常加班到 10 点。之前在美团实习过。
饮食：不吃香菜，不吃辣以外的刺激性食物。喜欢吃川菜，经常推荐成都小吃。最喜欢的餐厅是望京的某家火锅店。
音乐：最近在学吉他，学了三个月了，能弹几首简单的歌。想组个乐队但还没找到人。喜欢听摇滚和民谣。最喜欢的乐队是新裤子。
宠物：养了一只橘猫叫花花，两岁了。经常在朋友圈发猫的照片。花花最近生病了，带去看了兽医。
家庭：老家在成都，父母都是老师。有一个妹妹在读研。过年都会回成都。
感情：目前单身，之前谈过一个女朋友，分手一年了。最近好像对部门新来的设计师有好感。
旅行：喜欢旅行，去过日本（东京、京都）、泰国（清迈）。计划明年去新西兰。喜欢拍照，最近买了一个富士相机。
其他：生日是 3 月 15 号。喜欢看科幻电影，最喜欢的是星际穿越。最近在追一部韩剧。打游戏比较少，偶尔打王者荣耀。开一辆白色的特斯拉 Model 3。"""

print(f"短记忆：{len(memory_short)} 字")
print(f"中记忆：{len(memory_medium)} 字")
print(f"长记忆：{len(memory_long)} 字")

In [ ]:
def test_memory_injection(memory, label):
    """用不同长度的记忆测试回复质量"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": f"""你是帮用户回复消息的助手。语气自然随意，像朋友聊天。

以下是关于这个联系人的信息：
{memory}

请生成 3 条回复建议。"""
            },
            {"role": "user", "content": "帮我回复：'周末去爬山吗？'"}
        ]
    )
    usage = response.usage
    print(f"===== {label} =====")
    print(f"Token 用量：输入 {usage.prompt_tokens} + 输出 {usage.completion_tokens} = {usage.total_tokens}")
    print(response.choices[0].message.content)
    print()


test_memory_injection(memory_short, "短记忆 (~100字)")
test_memory_injection(memory_medium, "中记忆 (~500字)")
test_memory_injection(memory_long, "长记忆 (~2000字)")

### 观察要点

1. **Token 用量**：长记忆的输入 token 明显更多，但回复质量不一定更好
2. **相关性**：中等长度的记忆往往回复质量最好——信息刚好够用
3. **干扰**：长记忆里的"单身"、"韩剧"等信息跟爬山无关，但 AI 有时会莫名提到

**结论：关键不是信息多，而是信息"相关"。** 500 字的精准记忆 > 2000 字的全量记忆。

---

## 6.4 分层注入策略

ChatGPT 的 prompt 其实有多层结构：

```
┌──────────────────────────────────┐
│ 第 1 层：系统人设                │  "你是 ChatGPT，一个有帮助的 AI 助手..."
├──────────────────────────────────┤
│ 第 2 层：用户偏好/记忆            │  "用户喜欢简洁回复，是一名程序员..."
├──────────────────────────────────┤
│ 第 3 层：对话上下文              │  最近几轮的对话历史
├──────────────────────────────────┤
│ 第 4 层：当前消息                │  用户刚发的那条消息
└──────────────────────────────────┘
```

我们的键盘 App 也可以借鉴这个思路：

```
┌──────────────────────────────────┐
│ 第 1 层：系统指令                │  "你是帮用户回复消息的助手..."
├──────────────────────────────────┤
│ 第 2 层：联系人 Profile           │  姓名、关系、关键特征
├──────────────────────────────────┤
│ 第 3 层：最近对话摘要             │  最近几次聊天的关键信息
├──────────────────────────────────┤
│ 第 4 层：当前消息                │  对方刚发的那条消息
└──────────────────────────────────┘
```

In [ ]:
def build_prompt(contact_name, contact_profile, recent_summary, incoming_message):
    """
    分层组装 prompt。

    Args:
        contact_name: 联系人姓名
        contact_profile: 联系人画像（姓名、关系、特征）
        recent_summary: 最近对话摘要
        incoming_message: 对方发来的消息

    Returns:
        可直接传给 OpenAI API 的 messages 列表
    """
    # 第 1 层：系统指令
    system_instruction = (
        "你是一个帮用户生成聊天回复的智能键盘助手。\n"
        "要求：\n"
        "- 语气自然随意，像用户本人在说话\n"
        "- 生成 3 条不同风格的回复建议\n"
        "- 回复要简短，适合手机聊天场景\n"
    )

    # 第 2 层：联系人 Profile
    profile_section = f"\n【联系人信息】\n{contact_profile}"

    # 第 3 层：最近对话摘要
    summary_section = ""
    if recent_summary:
        summary_section = f"\n【最近对话摘要】\n{recent_summary}"

    # 组装 system prompt
    system_prompt = system_instruction + profile_section + summary_section

    # 第 4 层：当前消息
    user_prompt = f"{contact_name} 发来消息：'{incoming_message}'\n请生成回复建议。"

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


# 测试一下
messages = build_prompt(
    contact_name="小明",
    contact_profile="""姓名：小明
关系：大学同学，关系很好
特征：喜欢爬山跑步，在字节跳动工作，养了一只猫叫花花""",
    recent_summary="上周聊过周末计划，他说最近加班很多想出去放松。上个月一起爬了香山。",
    incoming_message="周末去爬山吗？",
)

# 先看看组装好的 prompt 长什么样
print("===== 组装好的 prompt =====")
for msg in messages:
    print(f"\n[{msg['role']}]")
    print(msg["content"])

In [ ]:
# 调用 API 看看效果
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)

print("===== AI 回复 =====")
print(response.choices[0].message.content)
print(f"\nToken 用量：{response.usage.total_tokens}")

### build_prompt 的好处

1. **结构清晰**：每一层的职责分明，修改方便
2. **灵活组装**：没有对话摘要就不加，不会出现空白段落
3. **可测试**：可以单独测试每一层的效果
4. **可复用**：不同场景换不同的 system_instruction 就行

---

## 6.5 你们系统的记忆注入

你们 iOS 键盘 App 的后端服务 `keyboard_service_chat_v4.py` 里，prompt 组装逻辑就是类似的分层结构。

系统里的 `memory_provider` 支持三种模式：

| 模式 | 说明 | 适用场景 |
|------|------|----------|
| `null` | 不注入任何记忆 | 陌生联系人、测试 |
| `fact_only` | 只注入 fact 列表 | 记忆量少的联系人 |
| `layered` | 分层注入（profile + facts + summary） | 记忆丰富的联系人 |

我们用代码模拟这三种模式：

In [ ]:
# 模拟数据库里存的记忆
CONTACT_FACTS = {
    "小明": [
        "大学同学，计算机系",
        "喜欢爬山和跑步",
        "在字节跳动做后端开发",
        "养了一只橘猫叫花花",
        "不吃香菜",
        "老家成都",
    ],
}

CONTACT_PROFILES = {
    "小明": "小明，男，大学同学，关系很好，经常一起运动和吃饭。性格开朗，做事靠谱。",
}

CONTACT_SUMMARIES = {
    "小明": "上周讨论了周末计划，他说加班太多想放松。上个月一起爬了香山，走的野路线。",
}


def memory_provider(contact_name, mode):
    """
    根据模式返回不同的记忆文本。

    模拟 keyboard_service_chat_v4.py 中的 memory_provider 逻辑。
    mode: "null" | "fact_only" | "layered"
    """
    if mode == "null":
        return ""

    if mode == "fact_only":
        facts = CONTACT_FACTS.get(contact_name, [])
        if not facts:
            return ""
        return "关于此联系人的已知信息：\n" + "\n".join(f"- {f}" for f in facts)

    if mode == "layered":
        parts = []

        profile = CONTACT_PROFILES.get(contact_name)
        if profile:
            parts.append(f"【联系人画像】\n{profile}")

        facts = CONTACT_FACTS.get(contact_name, [])
        if facts:
            parts.append("【记忆要点】\n" + "\n".join(f"- {f}" for f in facts))

        summary = CONTACT_SUMMARIES.get(contact_name)
        if summary:
            parts.append(f"【最近对话摘要】\n{summary}")

        return "\n\n".join(parts)

    return ""


# 看看三种模式输出什么
for mode in ["null", "fact_only", "layered"]:
    print(f"\n{'=' * 40}")
    print(f"模式：{mode}")
    print("=" * 40)
    result = memory_provider("小明", mode)
    print(result if result else "（空，不注入任何记忆）")

In [ ]:
def keyboard_service_chat(contact_name, incoming_message, mode="layered"):
    """
    模拟 keyboard_service_chat_v4.py 的核心流程。

    1. 根据 mode 获取记忆
    2. 组装 prompt
    3. 调用 LLM
    """
    # 获取记忆
    memory_text = memory_provider(contact_name, mode)

    # 组装 system prompt
    system_prompt = (
        "你是一个帮用户生成聊天回复的智能键盘助手。\n"
        "语气自然随意，像用户本人在说话。\n"
        "请生成 3 条不同风格的回复建议。"
    )

    if memory_text:
        system_prompt += f"\n\n{memory_text}"

    # 调用 LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{contact_name} 发来消息：'{incoming_message}'\n请生成回复建议。"},
        ],
    )

    return response.choices[0].message.content


# 对比三种模式的输出
incoming = "周末去爬山吗？"

for mode in ["null", "fact_only", "layered"]:
    print(f"\n{'=' * 50}")
    print(f"模式：{mode}  |  消息：{incoming}")
    print("=" * 50)
    print(keyboard_service_chat("小明", incoming, mode))

### 三种模式的差异

| 模式 | 回复特点 | Token 成本 |
|------|---------|----------|
| `null` | 回复很泛，像回复陌生人 | 最低 |
| `fact_only` | 会提到爬山、字节等关键词，但缺乏上下文 | 中等 |
| `layered` | 会提到香山、最近加班等时间线信息，最自然 | 最高 |

在实际系统中，新联系人用 `null`，有了几条记忆后升级到 `fact_only`，记忆丰富后再用 `layered`。

---

## 6.6 本课小结

### 核心概念

| 概念 | 你学到了什么 |
|------|------------|
| RAG | 检索增强生成：先搜相关信息，再给 AI 生成回复 |
| 全量注入 | 把所有记忆塞进 prompt，简单但浪费 token |
| 检索后注入 | 只注入相关记忆，省 token 但需要检索系统 |
| Lost in the Middle | 信息太多时 AI 会忽略中间部分 |
| 分层注入 | 把 prompt 分成系统指令/画像/摘要/消息四层 |
| memory_provider | 你们系统的三种模式：null / fact_only / layered |

### 你们系统当前的方案

```
经典 RAG:    用户消息 → 向量检索 → 召回相关记忆 → 注入 prompt → 生成
你们系统:    用户消息 → 按联系人 ID 查 → 取出全部记忆 → 注入 prompt → 生成
                                ↑
                        这里没有"检索"这一步
                     是直接按联系人 ID 查出来全塞进去
```

严格来说，你们目前用的是**"直接注入"**而不是经典 RAG。因为没有根据消息内容做语义检索，而是按联系人 ID 查出所有记忆直接塞。

这在记忆量小的阶段完全够用。等每个联系人的记忆超过几十条时，就需要考虑加入真正的检索环节了。

### 下节预告

下一课我们会学习向量（Embedding）和相似度搜索——这是做"真正的 RAG"的基础技术。